In [1]:
!pip install python-dotenv


In [2]:

import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')



True

In [3]:
import base64
import json
import os
import time
from pathlib import Path

import requests


class IMGBBResponse:
    def __init__(self, data, success, status):
        self.data = data
        self.success = success
        self.status = status


cache = {}


def upload_image_to_imgbb(image_base64, name, api_key):
    if cache.get(name):
        return cache.get(name)

    url = "https://api.imgbb.com/1/upload"
    payload = {"key": api_key, "image": image_base64, "name": name}

    try:
        response = requests.post(url, data=payload, timeout=60)
        response_data = response.json()

        if not response_data.get("success"):
            message = response_data.get("error", {}).get("message", "No error message")
            print(f"Error {response.status_code}: {message}")
            return {
                "result": False,
                "error": message,
                "rate_limited": "rate limit" in message.lower(),
                "data": IMGBBResponse(response_data.get("data"), False, response.status_code),
            }

        result = {
            "result": True,
            "error": "",
            "rate_limited": False,
            "data": IMGBBResponse(response_data.get("data"), True, response.status_code),
        }
        cache[name] = result
        return result

    except Exception as e:
        print(f"Error: {str(e)}")
        return {
            "result": False,
            "error": str(e),
            "rate_limited": False,
            "data": IMGBBResponse(None, False, 503),
        }


def get_thumb_url(response_data):
    thumb = response_data.get("thumb")
    if isinstance(thumb, dict):
        return thumb.get("url") or None
    return thumb or None


In [4]:

# workspace_root = folder that holds home_manifest.json (repo root).
# Defaults to the notebook's working dir so it runs on any machine; override if needed.
workspace_root = Path.cwd()
manifest_path = workspace_root / "home_manifest.json"
assert manifest_path.exists(), f"home_manifest.json not found at {manifest_path}"


In [5]:
# progress bar
import sys
from ipywidgets import IntProgress, Text
from IPython.display import display

from IPython.display import clear_output



In [6]:
def find_local_thumb_path(recipe):
    thumb = recipe.get("thumb")
    if thumb:
        candidate = workspace_root / thumb
        if candidate.exists():
            return candidate

    recipe_id = recipe.get("id")
    thumbs_dir = workspace_root / "home" / "recipe_thumbs"
    for ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        candidate = thumbs_dir / f"{recipe_id}{ext}"
        if candidate.exists():
            return candidate

    for candidate in sorted(thumbs_dir.glob(f"{recipe_id}*")):
        if candidate.is_file():
            return candidate

    return None


def write_manifest(manifest):
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
        handle.write("\n")


def set_thumb_urls(recipe, imgbb_url):
    # imgbb first (direct i.ibb.co link), existing mirrors (github raw) kept as fallback.
    existing = [u for u in recipe.get("thumbUrls", []) if u and u != imgbb_url]
    recipe["thumbUrls"] = [imgbb_url] + existing


def load_api_keys():
    keys = []
    for name in ["IMGBB_API_1", "IMGBB_API_2", "IMGBB_API_3", "IMGBB_API_KEY"]:
        k = os.environ.get(name, "").strip()
        if k and k not in keys:
            keys.append(k)
    assert keys, "No imgbb key in .env (set IMGBB_API_KEY or IMGBB_API_1/2/3)"
    print(f"{len(keys)} distinct imgbb key(s) loaded")
    return keys


class RateLimited(Exception):
    pass


def upload_recipe_thumbs(manifest, delay_seconds=1, force=False):
    recipes = manifest.get("recipes", [])
    total_count = len(recipes)
    uploaded_count = 0

    global_count = IntProgress(min=0, max=total_count)
    global_count_text = Text(value=f'Total recipes: {total_count}', disabled=True)
    text = Text(value='Starting...', disabled=True)
    display(global_count_text)
    display(global_count)
    display(text)

    api_keys = load_api_keys()

    for recipe in recipes:
        recipe_id = recipe.get("id")

        if not force and recipe.get("imgbb"):
            set_thumb_urls(recipe, recipe["imgbb"])
            text.value = f"Skip {recipe_id}: already uploaded"
            global_count.value += 1
            continue

        thumb_path = find_local_thumb_path(recipe)
        if not thumb_path:
            text.value = f"Skip {recipe_id}: local thumb not found"
            global_count.value += 1
            continue

        image_base64 = base64.b64encode(thumb_path.read_bytes()).decode("utf-8")

        succeeded = False
        for key in api_keys:
            result = upload_image_to_imgbb(image_base64, thumb_path.name, key)
            if result.get("result"):
                data = getattr(result.get("data"), "data", None) or {}
                imgbb_url = data.get("url")
                recipe["imgbb"] = imgbb_url
                recipe["imgbb_display_url"] = data.get("display_url")
                recipe["imgbb_thumb"] = get_thumb_url(data)
                recipe.pop("imgbb_error", None)
                set_thumb_urls(recipe, imgbb_url)
                text.value = f"Uploaded {recipe_id} -> {imgbb_url}"
                succeeded = True
                break
            elif result.get("rate_limited"):
                # every distinct key is rate limited -> stop; rerun after cooldown resumes.
                if key == api_keys[-1]:
                    write_manifest(manifest)
                    raise RateLimited(
                        f"All {len(api_keys)} key(s) rate limited after {uploaded_count} uploads. "
                        "Wait for the imgbb quota window to reset (or add more distinct keys "
                        "as IMGBB_API_1/2/3 in .env), then rerun this cell to resume."
                    )
            else:
                text.value = f"Failed {recipe_id} on a key: {result.get('error')}"

        if not succeeded:
            recipe["imgbb_error"] = "upload failed"

        write_manifest(manifest)
        global_count.value += 1
        global_count_text.value = f'Total recipes: {total_count}, Processed: {global_count.value}'
        if succeeded:
            uploaded_count += 1
        time.sleep(delay_seconds)

    print(f"Finished. Uploaded {uploaded_count} new thumbs (of {total_count} recipes).")
    return manifest


In [7]:

with manifest_path.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

# force=False resumes (skips recipes already carrying an "imgbb" url).
# Set force=True to re-upload everything.
upload_recipe_thumbs(manifest, delay_seconds=1, force=False)


Text(value='Total recipes: 88', disabled=True)

IntProgress(value=0, max=88)

Text(value='Starting...', disabled=True)

1 distinct imgbb key(s) loaded
Finished. Uploaded 88 new thumbs (of 88 recipes).


{'version': 6,
 'heroRotationMs': None,
 'heroPool': ['MOSAIC', 'DAILY_EDIT', 'WEEK', 'MOOD'],
 'stories': [{'id': 'story_reshape',
   'label': 'Reshape',
   'icon': 'style_icons/Abstract/A3.webp',
   'tool': 'RESHAPE',
   'addedMs': 1784419200000},
  {'id': 'story_portrait',
   'label': 'Portrait',
   'icon': 'style_icons/Classic/C24.webp',
   'tool': 'PORTRAIT',
   'addedMs': 1784332800000},
  {'id': 'story_draw',
   'label': 'Draw',
   'icon': 'style_icons/Drawing/D8.webp',
   'tool': 'DRAW',
   'addedMs': 1784246400000},
  {'id': 'story_selective',
   'label': 'Selective',
   'icon': 'style_icons/Landscape/LS-22.webp',
   'tool': 'SELECTIVE',
   'addedMs': 1784160000000},
  {'id': 'story_background',
   'label': 'Background',
   'icon': 'style_icons/Winter/W3.webp',
   'tool': 'BACKGROUND',
   'addedMs': 1784073600000},
  {'id': 'story_blend',
   'label': 'Blend',
   'icon': 'style_icons/Bloom/B4.webp',
   'tool': 'BLEND',
   'addedMs': 1783987200000},
  {'id': 'story_double_exposu